In [ ]:
# Import pandas for data handling and display
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import plotly express for bar charts and histograms
import plotly.express as px

# Import plotly graph objects for custom plots (ROC curves)
import plotly.graph_objects as go

# Import train_test_split for splitting data into train and test sets
from sklearn.model_selection import train_test_split, GridSearchCV

# Import StandardScaler to normalise feature values
from sklearn.preprocessing import StandardScaler

# Import Gaussian Naive Bayes classifier
from sklearn.naive_bayes import GaussianNB

# Import Logistic Regression classifier
from sklearn.linear_model import LogisticRegression

# Import K-Nearest Neighbours classifier
from sklearn.neighbors import KNeighborsClassifier

# Import all evaluation metrics needed for Task 5
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

print('All libraries imported successfully!')


All libraries imported successfully!


# Load the cleaned Classification Dataset

In [ ]:
df = pd.read_csv("../data/dataset1_classification_cleaned.csv")


# Drop any remaining missing values
df = df.dropna()

print('Dataset Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head()


Dataset Shape: (58628, 17)

Columns: ['age', 'income', 'employment_length', 'loan_amount', 'loan_interest_rate', 'loan_income_ratio', 'payment_default_on_file', 'credit_history_length', 'loan_approval_status', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']


,age,income,employment_length,loan_amount,loan_interest_rate,loan_income_ratio,payment_default_on_file,credit_history_length,loan_approval_status,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,21.0,12000,0.0,15000,6.99,0.12,0,4.0,0,0,1,0,1,0,0,0,0
1,21.0,13200,2.0,22500,16.77,0.19,1,3.0,0,0,1,0,1,0,0,0,0
2,23.0,9600,5.0,22500,12.42,0.31,0,3.0,0,0,0,1,0,0,1,0,0
3,40.0,126000,3.0,22500,8.00,0.19,0,11.0,0,0,0,1,1,0,0,0,0
4,40.0,90000,3.0,22500,12.42,0.39,0,14.0,0,0,0,0,0,1,0,0,0


#Target Variable Distribution

In [ ]:
# Target variable distribution
print('Target Variable Distribution — loan_approval_status:')
print(df['loan_approval_status'].value_counts())

fig = px.bar(
    x=['Rejected (0)', 'Approved (1)'],
    y=df['loan_approval_status'].value_counts().values,
    color=['Rejected (0)', 'Approved (1)'],
    color_discrete_sequence=['#EF553B', '#636EFA'],
    title='Target Class Distribution — Loan Approval Status',
    labels={'x': 'Approval Status', 'y': 'Count'}
)
fig.show()

Target Variable Distribution — loan_approval_status:
loan_approval_status
0    50281
1     8347
Name: count, dtype: int64


In [ ]:
# Define input features (X) and target output (y)
X = df.drop(columns=['loan_approval_status'])
y = df['loan_approval_status']

# Drop any rows where X still has NaN (can happen after one-hot encoding)
mask = X.notna().all(axis=1)
X = X[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True)

# --- SCREENSHOT THIS OUTPUT FOR TASK 4(b)(i) ---
print('Feature names used for building classification models:')
print(X.columns.tolist())
print('\nData shape (rows, features):', X.shape)
print('Target shape               :', y.shape)
print('NaNs remaining in X:', X.isnull().sum().sum())

Feature names used for building classification models:
['age', 'income', 'employment_length', 'loan_amount', 'loan_interest_rate', 'loan_income_ratio', 'payment_default_on_file', 'credit_history_length', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']

Data shape (rows, features): (58628, 16)
Target shape               : (58628,)
NaNs remaining in X: 0


#Task 4(b)(iii) - Train-Test Split(80/20)

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training set size:', X_train.shape)
print('Testing set size :', X_test.shape)

print('\nClass label ratio in Training set:')
print(y_train.value_counts())
print(y_train.value_counts(normalize=True).round(3))

print('\nClass label ratio in Test set:')
print(y_test.value_counts())
print(y_test.value_counts(normalize=True).round(3))

Training set size: (46902, 16)
Testing set size : (11726, 16)

Class label ratio in Training set:
loan_approval_status
0    40224
1     6678
Name: count, dtype: int64
loan_approval_status
0    0.858
1    0.142
Name: proportion, dtype: float64

Class label ratio in Test set:
loan_approval_status
0    10057
1     1669
Name: count, dtype: int64
loan_approval_status
0    0.858
1    0.142
Name: proportion, dtype: float64


#Feature Scaling - StandardScaler

In [ ]:
# Feature Scaling — StandardScaler (required for LR and KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('StandardScaler applied to training and test sets.')

StandardScaler applied to training and test sets.


#Reusable Evaluation Function

In [ ]:
# Define a reusable function to evaluate any classification model
def evaluate_model(model_name, y_true, y_pred, y_prob=None):
    # Calculate accuracy score
    acc  = accuracy_score(y_true, y_pred)
    # Calculate precision score
    prec = precision_score(y_true, y_pred, zero_division=0)
    # Calculate recall score
    rec  = recall_score(y_true, y_pred, zero_division=0)
    # Calculate F1 score
    f1   = f1_score(y_true, y_pred, zero_division=0)
    # Calculate AUC-ROC if probabilities are provided
    auc  = roc_auc_score(y_true, y_prob) if y_prob is not None else None

    # Print all metric results
    print(f'\n{"="*50}')
    print(f'  Model     : {model_name}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    if auc is not None:
        print(f'  ROC-AUC   : {auc:.4f}')
    print(f'{"="*50}')

    # Print full classification report with per-class metrics
    print('\nClassification Report:')
    print(classification_report(y_true, y_pred))

    # Return results as a dictionary for later comparison
    return {'Model': model_name, 'Accuracy': acc, 'Precision': prec,
            'Recall': rec, 'F1-Score': f1, 'ROC-AUC': auc}

# Initialise lists to collect results and probabilities for all models
all_results = []
all_probs   = {}
print('Evaluation function ready.')


Evaluation function ready.


# Task 4(a) - K Value Selection for K Nearest Neighbours(KNN)

In [ ]:
# Test K values from 1 to 30 to find the optimal number of neighbours
k_range = range(1, 31)
error_rates = []
accuracy_scores_k = []

# Loop through each K value and record error rate and accuracy
for k in k_range:
    # Initialise KNN with current K value
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    # Train on scaled training data
    knn_temp.fit(X_train_scaled, y_train)
    # Predict on scaled test data
    pred_temp = knn_temp.predict(X_test_scaled)
    # Record error rate (1 - accuracy)
    error_rates.append(1 - accuracy_score(y_test, pred_temp))
    # Record accuracy score
    accuracy_scores_k.append(accuracy_score(y_test, pred_temp))

# Plot error rate vs K value (Elbow Method) to identify optimal K
fig_elbow = go.Figure()
fig_elbow.add_trace(go.Scatter(
    x=list(k_range),
    y=error_rates,
    mode='lines+markers',
    marker=dict(color='crimson', size=8),
    line=dict(color='crimson'),
    name='Error Rate'
))
fig_elbow.update_layout(
    title='Task 4(a) — Elbow Method: Error Rate vs K Value',
    xaxis_title='K (Number of Neighbours)',
    yaxis_title='Error Rate',
    template='plotly_white'
)
fig_elbow.show()

# Select the K value with the lowest error rate
best_k = list(k_range)[error_rates.index(min(error_rates))]
print(f'Optimal K selected from Elbow Method: K = {best_k}')


Optimal K selected from Elbow Method: K = 7


#Task 4b - Model 1: Naive Bayes (NB)

In [ ]:
# Initialise Gaussian Naive Bayes classifier
# NB does not require feature scaling — uses raw unscaled data
nb_model = GaussianNB()

# Train Naive Bayes on unscaled training data
nb_model.fit(X_train, y_train)

# Predict class labels on unscaled test data
nb_pred = nb_model.predict(X_test)

# Get predicted probabilities for class 1 (Approved) for AUC-ROC
nb_prob = nb_model.predict_proba(X_test)[:, 1]

# Evaluate Naive Bayes model using the evaluation function
nb_results = evaluate_model('Naive Bayes (NB)', y_test, nb_pred, nb_prob)

# Store results and probabilities for later comparison
all_results.append(nb_results)
all_probs['NB'] = nb_prob



  Model     : Naive Bayes (NB)
  Accuracy  : 0.8865
  Precision : 0.7493
  Recall    : 0.3044
  F1-Score  : 0.4329
  ROC-AUC   : 0.8440

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.98      0.94     10057
           1       0.75      0.30      0.43      1669

    accuracy                           0.89     11726
   macro avg       0.82      0.64      0.68     11726
weighted avg       0.87      0.89      0.87     11726



# Task 4b - Model 2:Logistic Regression

In [ ]:
# Initialise Logistic Regression classifier
# max_iter=1000 ensures convergence; random_state=42 for reproducibility
# LR requires scaled features — uses X_train_scaled and X_test_scaled
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# Train Logistic Regression on scaled training data
lr_model.fit(X_train_scaled, y_train)

# Predict class labels on scaled test data
lr_pred = lr_model.predict(X_test_scaled)

# Get predicted probabilities for class 1 (Approved) for AUC-ROC
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate Logistic Regression model using the evaluation function
lr_results = evaluate_model('Logistic Regression (LR)', y_test, lr_pred, lr_prob)

# Store results and probabilities for later comparison
all_results.append(lr_results)
all_probs['LR'] = lr_prob


# Task 4b - Model 3: K- Nearest Neighbours (KNN)




In [ ]:
# Train KNN — uses scaled features
# K value chosen here — update K= based on your Task 4(a) hyperparameter choice
K = 5
knn_model = KNeighborsClassifier(n_neighbors=K)
knn_model.fit(X_train_scaled, y_train)

knn_pred = knn_model.predict(X_test_scaled)
knn_prob = knn_model.predict_proba(X_test_scaled)[:, 1]

knn_results = evaluate_model(f'KNN (K={K})', y_test, knn_pred, knn_prob)
all_results.append(knn_results)
all_probs[f'KNN (K={K})'] = knn_prob

print(f'\nTask 4(b) COMPLETE — All 3 models trained and evaluated.')


  Model     : KNN (K=5)
  Accuracy  : 0.9207
  Precision : 0.8302
  Recall    : 0.5566
  F1-Score  : 0.6664
  ROC-AUC   : 0.8590

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.98      0.95     10057
           1       0.83      0.56      0.67      1669

    accuracy                           0.92     11726
   macro avg       0.88      0.77      0.81     11726
weighted avg       0.92      0.92      0.91     11726


Task 4(b) COMPLETE — All 3 models trained and evaluated.


# Task 5a - Confusion Matrix

In [ ]:
# Confusion Matrix — Naive Bayes
cm_nb = confusion_matrix(y_test, nb_pred)
fig_nb = px.imshow(
    cm_nb,
    text_auto=True,
    color_continuous_scale='Blues',
    labels=dict(x='Predicted Label', y='Actual Label'),
    x=['Rejected (0)', 'Approved (1)'],
    y=['Rejected (0)', 'Approved (1)'],
    title='Task 5(a) — Confusion Matrix: Naive Bayes (NB)'
)
fig_nb.show()

In [ ]:
# Confusion Matrix — Logistic Regression model
cm_lr = confusion_matrix(y_test, lr_pred)
fig_lr = px.imshow(
    cm_lr,
    text_auto=True,
    color_continuous_scale='Greens',
    labels=dict(x='Predicted Label', y='Actual Label'),
    x=['Rejected (0)', 'Approved (1)'],
    y=['Rejected (0)', 'Approved (1)'],
    title='Task 5(a) — Confusion Matrix: Logistic Regression (LR)'
)
fig_lr.show()

In [ ]:
# Confusion Matrix — KNN
cm_knn = confusion_matrix(y_test, knn_pred)
fig_knn = px.imshow(
    cm_knn,
    text_auto=True,
    color_continuous_scale='Oranges',
    labels=dict(x='Predicted Label', y='Actual Label'),
    x=['Rejected (0)', 'Approved (1)'],
    y=['Rejected (0)', 'Approved (1)'],
    title=f'Task 5(a) — Confusion Matrix: KNN (K={K})'
)
fig_knn.show()

#Classification Report

In [ ]:
# Print full classification report for Naive Bayes model
print('Classification Report — Naive Bayes (NB):')
print('=' * 55)
# classification_report shows precision, recall, f1 and support for each class
print(classification_report(
    y_test,
    nb_pred,
    target_names=['Rejected (0)', 'Approved (1)'],
    zero_division=0
))

Classification Report — Naive Bayes (NB):
              precision    recall  f1-score   support

Rejected (0)       0.89      0.98      0.94     10057
Approved (1)       0.75      0.30      0.43      1669

    accuracy                           0.89     11726
   macro avg       0.82      0.64      0.68     11726
weighted avg       0.87      0.89      0.87     11726



In [ ]:
# Print full classification report for Logistic Regression model
print('Classification Report — Logistic Regression (LR):')
print('=' * 55)
# classification_report shows precision, recall, f1 and support for each class
print(classification_report(
    y_test,
    lr_pred,
    target_names=['Rejected (0)', 'Approved (1)'],
    zero_division=0
))

In [ ]:
# Print full classification report for KNN model
print(f'Classification Report — KNN (K={K}):')
print('=' * 55)
# classification_report shows precision, recall, f1 and support for each class
print(classification_report(
    y_test,
    knn_pred,
    target_names=['Rejected (0)', 'Approved (1)'],
    zero_division=0
))

# Task 5a - AuC-Roc curves(all 3 models on One Plot)





In [ ]:
# Plot AUC-ROC curves for all three models on a single graph
fig_roc = go.Figure()

# Define models, their probabilities and display colours
model_probs = {
    'Naive Bayes (NB)': nb_prob,
    'Logistic Regression (LR)': lr_prob,
    f'KNN (K={K})': knn_prob
}
colors = ['#636EFA', '#EF553B', '#00CC96']

# Loop through each model and add its ROC curve to the plot
for (name, prob), color in zip(model_probs.items(), colors):
    # Calculate false positive rate, true positive rate and thresholds
    fpr, tpr, _ = roc_curve(y_test, prob)
    # Calculate AUC score for this model
    auc_val = roc_auc_score(y_test, prob)
    # Add the ROC curve trace to the figure
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f'{name} (AUC = {auc_val:.3f})',
        line=dict(color=color, width=2)
    ))

# Add random classifier baseline (diagonal dashed line)
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random Classifier',
    line=dict(color='grey', width=1, dash='dash')
))

# Update chart layout
fig_roc.update_layout(
    title='Task 5(a) — AUC-ROC Curves: NB vs LR vs KNN',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    legend=dict(x=0.6, y=0.1),
    width=700, height=500,
    title_x=0.5
)
fig_roc.show()


#Task 5b - Evaluation Metrics Summary Table

In [ ]:
# Compile all model results into a summary DataFrame for Task 5(b)
results_df = pd.DataFrame(all_results).set_index('Model')

# Select and order the 5 required evaluation metrics
results_df = results_df[['Accuracy', 'Recall', 'Precision', 'F1-Score', 'ROC-AUC']]

# --- SCREENSHOT THIS TABLE FOR TASK 5(b) ---
print('Task 5(b) — All Evaluation Metrics for NB, LR and KNN:')
print(results_df.round(4).to_string())


Task 5(b) — All Evaluation Metrics for NB, LR and KNN:
                          Accuracy  Recall  Precision  F1-Score  ROC-AUC
Model                                                                   
Naive Bayes (NB)            0.8790  0.2834     0.6796    0.4000   0.8305
Naive Bayes (NB)            0.8790  0.2834     0.6796    0.4000   0.8305
Logistic Regression (LR)    0.8954  0.4248     0.7264    0.5361   0.8864
KNN (K=5)                   0.9200  0.5578     0.8232    0.6650   0.8610
KNN (K=5)                   0.9200  0.5578     0.8232    0.6650   0.8610


In [ ]:
# Visual bar chart of all metrics across all 3 models
results_melted = results_df.reset_index().melt(id_vars='Model', var_name='Metric', value_name='Score')

fig_metrics = px.bar(
    results_melted,
    x='Metric',
    y='Score',
    color='Model',
    barmode='group',
    title='Task 5(b) — Model Comparison: All Evaluation Metrics',
    color_discrete_sequence=['#636EFA', '#EF553B', '#00CC96'],
    range_y=[0, 1.1]
)
fig_metrics.show()

# Task 5(d)(i) - GridSearchCV Hyperparamter Tuning on Best Model

In [ ]:
# Define the hyperparameter grid for KNN tuning
# n_neighbors — number of neighbours to test
# metric — distance metric to use
# weights — weighting strategy for neighbours
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'metric': ['euclidean', 'manhattan'],
    'weights': ['uniform', 'distance']
}

# Initialise GridSearchCV with KNN estimator
# cv=5 — 5-fold cross validation for robust evaluation
# scoring='recall' — optimise for recall as per success criteria
# n_jobs=-1 — use all CPU cores for speed
grid_search = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid_knn,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

# Fit GridSearchCV on scaled training data
grid_search.fit(X_train_scaled, y_train)

# Print the best hyperparameters found by GridSearchCV
print('Task 5(d)(i) — Best Hyperparameters found by GridSearchCV:')
print(grid_search.best_params_)
print(f'\nBest CV Recall Score: {grid_search.best_score_:.4f}')


Task 5(d)(i) — Best Hyperparameters found by GridSearchCV:
{'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}

Best CV Recall Score: 0.5538


# Task 5d (ii) - Evaluate Best Model Before vs After Tuning

In [ ]:
# Retrieve the best model found by GridSearchCV
best_model_tuned = grid_search.best_estimator_

# Predict class labels using the tuned best model
best_pred_tuned = best_model_tuned.predict(X_test_scaled)

# Get predicted probabilities from the tuned model
best_prob_tuned = best_model_tuned.predict_proba(X_test_scaled)[:, 1]

# Evaluate tuned model performance using evaluation function
tuned_results = evaluate_model('Best Model (Tuned — GridSearchCV)', y_test, best_pred_tuned, best_prob_tuned)

# Print best hyperparameters for report
print('\nTask 5(d)(ii) — Best Hyperparameters:')
print(grid_search.best_params_)



  Model     : Best Model (Tuned — GridSearchCV)
  Accuracy  : 0.9128
  Precision : 0.7576
  Recall    : 0.5692
  F1-Score  : 0.6500
  ROC-AUC   : 0.8398

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.97      0.95     10057
           1       0.76      0.57      0.65      1669

    accuracy                           0.91     11726
   macro avg       0.84      0.77      0.80     11726
weighted avg       0.91      0.91      0.91     11726


Task 5(d)(ii) — Best Hyperparameters:
{'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}


In [ ]:
# Plot confusion matrix BEFORE tuning (KNN with best_k from elbow method)
cm_before = confusion_matrix(y_test, knn_pred)
fig_before = px.imshow(
    cm_before, text_auto=True,
    color_continuous_scale='Oranges',
    labels=dict(x='Predicted Label', y='Actual Label'),
    x=['Rejected (0)', 'Approved (1)'],
    y=['Rejected (0)', 'Approved (1)'],
    title=f'Task 5(d)(ii) — Confusion Matrix BEFORE Tuning: KNN (K={K})'
)
# Centre the title
fig_before.update_layout(title_x=0.5)
fig_before.show()


In [ ]:
# Plot confusion matrix AFTER tuning (best model from GridSearchCV)
cm_after = confusion_matrix(y_test, best_pred_tuned)
fig_after = px.imshow(
    cm_after, text_auto=True,
    color_continuous_scale='Purples',
    labels=dict(x='Predicted Label', y='Actual Label'),
    x=['Rejected (0)', 'Approved (1)'],
    y=['Rejected (0)', 'Approved (1)'],
    title='Task 5(d)(ii) — Confusion Matrix AFTER Tuning (GridSearchCV)'
)
# Centre the title
fig_after.update_layout(title_x=0.5)
fig_after.show()


In [ ]:
# --- SCREENSHOT THIS OUTPUT FOR TASK 5(d)(ii) ---
# Calculate and compare Recall, Precision, F1 before and after tuning
before_recall = recall_score(y_test, knn_pred, zero_division=0)
after_recall   = recall_score(y_test, best_pred_tuned, zero_division=0)
before_prec   = precision_score(y_test, knn_pred, zero_division=0)
after_prec     = precision_score(y_test, best_pred_tuned, zero_division=0)
before_f1      = f1_score(y_test, knn_pred, zero_division=0)
after_f1       = f1_score(y_test, best_pred_tuned, zero_division=0)

# Build comparison DataFrame
comparison_df = pd.DataFrame({
    'Metric'        : ['Recall', 'Precision', 'F1-Score'],
    'BEFORE Tuning' : [before_recall, before_prec, before_f1],
    'AFTER Tuning'  : [after_recall, after_prec, after_f1]
})

# Print comparison table
print('Task 5(d)(ii) — BEFORE vs AFTER Hyperparameter Tuning:')
print(comparison_df.round(4).to_string(index=False))


Task 5(d)(ii) — BEFORE vs AFTER Hyperparameter Tuning:
   Metric  BEFORE Tuning  AFTER Tuning
   Recall         0.5578        0.5692
Precision         0.8232        0.7576
 F1-Score         0.6650        0.6500


In [ ]:
# Visualise Before vs After comparison
fig_compare = px.bar(
    comparison_df.melt(id_vars='Metric', var_name='Stage', value_name='Score'),
    x='Metric',
    y='Score',
    color='Stage',
    barmode='group',
    title='Task 5(d)(ii) — Before vs After Hyperparameter Tuning',
    color_discrete_sequence=['#EF553B', '#636EFA'],
    range_y=[0, 1.1]
)
fig_compare.show()

print('\nTask 5(d) COMPLETE — GridSearchCV tuning done!')


Task 5(d) COMPLETE — GridSearchCV tuning done!


# Notebook 2 - Final Summary

In [ ]:
# Print complete summary of all results from Notebook 2
print('='*60)
print('  NOTEBOOK 2 — FINAL SUMMARY')
print('='*60)

# Task 4(b) summary
print('\nTASK 4(b) — Models Built:')
print(f'  - Naive Bayes (NB)          — unscaled data')
print(f'  - Logistic Regression (LR)  — scaled data')
print(f'  - KNN (K={K})               — scaled data, K from elbow method')

# Task 5(a) summary
print('\nTASK 5(a) — Outputs Generated:')
print('  - Confusion Matrix for each model')
print('  - Classification Report for each model')
print('  - AUC-ROC Curve for all 3 models on one plot')

# Task 5(b) metrics table
print('\nTASK 5(b) — Evaluation Metrics Summary:')
print(results_df.round(4).to_string())

# Task 5(d) summary
print('\nTASK 5(d) — GridSearchCV Best Hyperparameters:')
print(grid_search.best_params_)

print('\nAll results ready for the Analysis Report.')
print('='*60)
